In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

2026-05-28 17:11:51.247261: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779988311.470732      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779988311.537990      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779988312.090376      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779988312.090419      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779988312.090422      58 computation_placer.cc:177] computation placer alr

In [5]:
!ls /kaggle/input/datasets/trainingdatapro/aggressive-behavior-video-classification

aggressive_behavior.csv  files


In [6]:
DATASET_PATH = "/kaggle/input/datasets/trainingdatapro/aggressive-behavior-video-classification/files"

IMG_SIZE = 112
SEQUENCE_LENGTH = 10
BATCH_SIZE = 2
EPOCHS = 30

print(os.listdir(DATASET_PATH))

['aggressive', 'non_aggressive']


In [7]:
CLASS_NAMES = ["aggressive", "non_aggressive"]

for cls in CLASS_NAMES:
    print(cls, len(os.listdir(os.path.join(DATASET_PATH, cls))))

aggressive 5
non_aggressive 6


In [8]:
def extract_frames(video_path):
    frames = []
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, total_frames - 1, SEQUENCE_LENGTH).astype(int)

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()

        if success:
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = frame / 255.0
            frames.append(frame)

    cap.release()

    if len(frames) == SEQUENCE_LENGTH:
        return np.array(frames)

    return None

In [9]:
videos = []
labels = []

for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)

    for video_name in os.listdir(class_path):
        if not video_name.endswith(".mp4"):
            continue

        video_path = os.path.join(class_path, video_name)
        frames = extract_frames(video_path)

        if frames is not None:
            videos.append(frames)
            labels.append(class_name)

videos = np.array(videos)
labels = np.array(labels)

print("Videos shape:", videos.shape)
print("Labels shape:", labels.shape)
print("Classes:", np.unique(labels))

Videos shape: (11, 10, 112, 112, 3)
Labels shape: (11,)
Classes: ['aggressive' 'non_aggressive']


In [10]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

num_classes = len(encoder.classes_)
labels_categorical = tf.keras.utils.to_categorical(labels_encoded, num_classes)

print("Classes:", encoder.classes_)
print("Encoded labels:", labels_encoded)

Classes: ['aggressive' 'non_aggressive']
Encoded labels: [0 0 0 0 0 1 1 1 1 1 1]


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    videos,
    labels_categorical,
    test_size=0.2,
    random_state=42,
    stratify=labels_encoded
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8, 10, 112, 112, 3)
X_test: (3, 10, 112, 112, 3)


In [12]:
model = models.Sequential([
    layers.TimeDistributed(
        layers.Conv2D(32, (3, 3), activation='relu'),
        input_shape=(SEQUENCE_LENGTH, IMG_SIZE, IMG_SIZE, 3)
    ),
    layers.TimeDistributed(layers.MaxPooling2D((2, 2))),

    layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation='relu')),
    layers.TimeDistributed(layers.MaxPooling2D((2, 2))),

    layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation='relu')),
    layers.TimeDistributed(layers.MaxPooling2D((2, 2))),

    layers.TimeDistributed(layers.Flatten()),

    layers.LSTM(128),
    layers.Dropout(0.5),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1779988453.334500      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779988453.340391      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 10, 110, 110,   │           896 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 10, 55, 55, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 10, 53, 53, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 10, 26, 26, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 10, 24, 24,     │        73,856 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 10, 12, 12,     │             0 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 10, 18432)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │     9,503,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,604,866 (36.64 MB)

 Trainable params: 9,604,866 (36.64 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=BATCH_SIZE
)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 1.0000 - loss: 6.8050e-04 - val_accuracy: 0.6667 - val_loss: 0.9632
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 1.0000 - loss: 4.3308e-04 - val_accuracy: 0.6667 - val_loss: 0.9887
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 1.0000 - loss: 0.0041 - val_accuracy: 0.6667 - val_loss: 0.8570
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 8.6604e-04 - val_accuracy: 0.6667 - val_loss: 0.9276
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 1.0000 - loss: 0.0167 - val_accuracy: 0.6667 - val_loss: 1.3099
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 2.0344e-04 - val_accuracy: 0.6667 - val_loss: 1.3158
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 3.5597e-04 - val_accuracy: 0.6667 - val_loss: 1.2644
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.6667

In [20]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6667 - loss: 2.7158
Test Loss: 2.7158071994781494
Test Accuracy: 0.6666666865348816


In [21]:
def predict_video(video_path):
    frames = extract_frames(video_path)

    if frames is None:
        print("Could not read video")
        return

    frames = np.expand_dims(frames, axis=0)

    prediction = model.predict(frames)
    predicted_index = np.argmax(prediction)
    confidence = np.max(prediction)

    print("Predicted Class:", encoder.classes_[predicted_index])
    print("Confidence:", confidence)

In [22]:
test_video_path = "/kaggle/input/datasets/trainingdatapro/aggressive-behavior-video-classification/files/non_aggressive/0.mp4"
predict_video(test_video_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Predicted Class: non_aggressive
Confidence: 0.9999745
